In [50]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [51]:
train_df = pd.read_csv('/kaggle/input/datasets/harshtruth/trainset/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/harshtruth/testset/test.csv')
print(train_df.shape)
train_df.head()

(10500, 10)


,ID,Age,Annual_Income,Credit_Score,Loan_Amount,Debt_to_Income,Employment_Years,Previous_Defaults,Education_Level,is_default
0,16868,31,74956.384288,696,30521.270737,0.249670,8,0,0,0
1,16107,58,53338.688256,496,27132.886897,0.493917,27,0,3,1
2,14367,46,15000.000000,783,4301.476830,0.241996,34,0,3,0
3,23009,35,57895.729791,738,39988.650926,0.574912,40,0,0,0
4,14539,56,71309.519094,785,16439.459217,0.658744,13,0,1,0


In [52]:
FEATURES = ['Age', 'Annual_Income', 'Credit_Score', 'Loan_Amount',
            'Debt_to_Income', 'Employment_Years', 'Previous_Defaults', 'Education_Level']

X = train_df[FEATURES].values.astype(np.float32)  # ← explicit cast
y = train_df['is_default'].values.astype(np.float32)  # ← explicit cast
X_test = test_df[FEATURES].values.astype(np.float32)

# Quick sanity check
print("y unique values:", np.unique(y))          # should be [0. 1.]
print("X shape:", X.shape)                        # should be (10500, 8)
print("X sample:\n", X[:3])

# Scale features
scaler = StandardScaler()
X      = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

print("\ny_train_t range:", y_train_t.min().item(), "–", y_train_t.max().item())  # must be 0.0 – 1.0
print("y_train_t dtype:", y_train_t.dtype)  # must be torch.float32

y unique values: [0. 1.]
X shape: (10500, 8)
X sample:
 [[3.1000000e+01 7.4956383e+04 6.9600000e+02 3.0521271e+04 2.4966979e-01
  8.0000000e+00 0.0000000e+00 0.0000000e+00]
 [5.8000000e+01 5.3338688e+04 4.9600000e+02 2.7132887e+04 4.9391747e-01
  2.7000000e+01 0.0000000e+00 3.0000000e+00]
 [4.6000000e+01 1.5000000e+04 7.8300000e+02 4.3014771e+03 2.4199601e-01
  3.4000000e+01 0.0000000e+00 3.0000000e+00]]

y_train_t range: 0.0 – 1.0
y_train_t dtype: torch.float32


In [53]:
class LoanMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),          
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()             
        )

    def forward(self, x):
        return self.network(x)

model = LoanMLP(input_dim=len(FEATURES))
print(model)

LoanMLP(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


In [54]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 100 

for epoch in range(EPOCHS):
    model.train()
    preds = model(X_train_t)
    loss  = criterion(preds, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t)
            val_loss  = criterion(val_preds, y_val_t)
        print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}')

Epoch 10/100 | Train Loss: 0.6540 | Val Loss: 0.6499
Epoch 20/100 | Train Loss: 0.6059 | Val Loss: 0.5981
Epoch 30/100 | Train Loss: 0.5444 | Val Loss: 0.5316
Epoch 40/100 | Train Loss: 0.4672 | Val Loss: 0.4513
Epoch 50/100 | Train Loss: 0.3897 | Val Loss: 0.3702
Epoch 60/100 | Train Loss: 0.3249 | Val Loss: 0.3024
Epoch 70/100 | Train Loss: 0.2785 | Val Loss: 0.2534
Epoch 80/100 | Train Loss: 0.2431 | Val Loss: 0.2206
Epoch 90/100 | Train Loss: 0.2159 | Val Loss: 0.1997
Epoch 100/100 | Train Loss: 0.2059 | Val Loss: 0.1869


In [55]:
model.eval()  
with torch.no_grad():
    val_probs = model(X_val_t).detach().numpy().flatten()

val_preds = (val_probs >= 0.5).astype(int)

print(f'Validation Accuracy : {accuracy_score(y_val, val_preds):.4f}')
print(f'Validation F1 Score : {f1_score(y_val, val_preds):.4f}')
print(f'Predicted default rate: {val_preds.mean():.3f}')

Validation Accuracy : 0.9238
Validation F1 Score : 0.9305
Predicted default rate: 0.553


In [56]:
model.eval()  
with torch.no_grad():
    test_probs = model(X_test_t).detach().numpy().flatten()

print(f'Prob range: {test_probs.min():.3f} – {test_probs.max():.3f}')
print(f'Predicted default rate: {(test_probs >= 0.5).mean():.3f}')

submission = pd.DataFrame({
    'ID'        : test_df['ID'],
    'is_default': (test_probs >= 0.5).astype(int)
})

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
submission.head()

Prob range: 0.001 – 1.000
Predicted default rate: 0.542
Saved submission.csv


,ID,is_default
0,20261,1
1,10770,0
2,15050,0
3,11773,0
4,18588,0
